# CellNeighborEX v2 Tutorial — 03. Identifying Spatial Region-Specific CCI Genes in Mouse Liver Cancer (HCC)

**Key concept:** Tumor microenvironments exhibit spatially organized programs, serving as a suitable biological system for investigating spatial region-specific CCI genes and interacting cell-type pairs.

This notebook starts from bundled **precomputed `ccisignal` outputs**. We therefore skip the deconvolution step and focus on identifying spatial region-specific CCI genes and their interaction cell type pairs using `ccigenes` and `ccipairs` modules.

## Core Concept: Interpreting Spatial CCI Signals in the Tumor Microenvironment

**The tumor microenvironment (TME) exhibits spatially organized cellular composition, making it an exemplary case study for examining how spatial region-specific CCI genes and their interacting cell type pairs can be interpreted in a spatial context.**

### Using mouse liver cancer spatial transcriptomics data as an example, this tutorial demonstrates how to:

- Examine spatial cell-type distributions across tumor-associated regions
- Identify spatial region-specific CCI genes
- Infer interacting cell type pairs for each CCI gene

In [ ]:
import os
import sys
from pathlib import Path

# --- Environment Configuration ---
# Set environment variables to manage threading for linear algebra libraries
# This helps prevent CPU over-subscription and ensures consistent performance
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

# --- Standard Library Imports ---
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

# --- Path Management ---
# Define and resolve the absolute paths for the project directory structure
RUN_CODE_DIR = Path("/home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/run_code").resolve()
BASE_DIR = RUN_CODE_DIR.parent  # Points to the main project root (CNEXv2)

# Add the base directory to sys.path to allow importing local modules
sys.path.insert(0, str(BASE_DIR))

# --- Local Module Imports (CellNeighborEX Package) ---

# Module for cell-cell interaction (CCI) signal processing and spatial clustering
from CellNeighborEX.ccisignal import compute_proportion, cluster_spots_by_proportion

# Module for identifying and statistically validating CCI-related genes
from CellNeighborEX.ccigenes import (
    clean_column_names,
    obtain_clean_celltype_names,
    permutation_test_all_clusters,
    chi_square_goodness_of_fit,
    compute_combined_p_value,
    simplify_gene_names,
)

# Module for modeling interaction terms and validating against databases
from CellNeighborEX.ccipairs import (
    regress_residual_on_interaction_with_regularization,
    extract_interaction_terms,
    compare_database,
)

In [ ]:
# --- Data Directory Configuration ---
# Set the path for liver cancer tutorial example data
DATA_DIR = BASE_DIR / "Tutorial-ExampleData" / "3_liver_cancer"
# Single-cell RNA-seq reference file path
SC_REF_FILE = DATA_DIR / "sc_ccisignal.h5ad"
# Spatial Visium data file path
VISIUM_FILE = DATA_DIR / "sp_ccisignal.h5ad"

# --- Metadata and Analysis Settings ---
SPECIES = "mouse"
# Key for the cluster information in the AnnData object (leiden clustering on proportions)
CLUSTER_INFO = "proportion_leiden"

# --- Tutorial Execution Parameters (Speed Knobs) ---
# Number of permutations for statistical significance testing
N_PERMUTATIONS = 250
# Log Fold Change threshold for differential expression analysis
LOG_FC = 0.5
# P-value threshold for interaction terms
PVAL_TERM = 0.05

# --- Output and Figure Directory Setup ---
# Create the main output directory for liver cancer results
OUTPUT_DIR = RUN_CODE_DIR / "tutorial_output" / "03_liver_cancer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create a subdirectory specifically for saving generated plots
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# --- Scanpy Plotting Settings ---
# Set the global figure directory for Scanpy's plotting functions
sc.settings.figdir = str((OUTPUT_DIR / "figures").resolve())
# Set global resolution (dots per inch) for figures
sc.settings.set_figure_params(dpi=120)

# --- Path Verification Logging ---
# Print paths to ensure all directories and files are correctly mapped
print(f"SC_REF_FILE : {SC_REF_FILE}")
print(f"VISIUM_FILE : {VISIUM_FILE}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

## 1) Load data and check metadata information

The following datasets are loaded:
- `sc_ccisignal.h5ad`: reference signatures
- `sp_ccisignal.h5ad`: spatial deconvolution map

In [ ]:
# --- Load Data ---
# Load the single-cell RNA-seq reference data (H5AD format)
adata_ref_sig = sc.read_h5ad(SC_REF_FILE)
# Load the spatial Visium data (H5AD format)
adata_vis = sc.read_h5ad(VISIUM_FILE)

# --- Summary Information ---
# Print the structure and metadata summary for both datasets
print("Reference:", adata_ref_sig)
print("Spatial:  ", adata_vis)

# --- Validation ---
# Ensure that posterior summaries (Cell2location results) are present in the .uns['mod'] slot
# These summaries are critical for downstream CellNeighborEX analysis
assert "mod" in adata_ref_sig.uns and "mod" in adata_vis.uns

# --- Metadata Standardization ---
# Check if the 'sample' column exists; if not, extract the sample name 
# from the spatial metadata dictionary to ensure consistent indexing
if "sample" not in adata_vis.obs.columns and "spatial" in adata_vis.uns:
    adata_vis.obs["sample"] = list(adata_vis.uns["spatial"].keys())[0]

# --- Preview ---
# Display the first few rows of the observation (spot) metadata for verification
display(adata_vis.obs.head())

## 2) (Optional) Recompute proportions and recluster 

The spatial region clustering results (`proportion_leiden`) are already precomputed and stored in the `adata_vis.obs` of the `sp_ccisignal.h5ad` file. Therefore, you do not need to run this step. However, when starting a CellNeighborEX analysis from scratch without precomputed outputs, you should perform this step to define spatial regions (niches). This process clusters spots based on their cell-type deconvolution results (proportions) to identify distinct biological domains within the tissue.

In [ ]:
# --- Spatial Proportions and Clustering ---

# Calculate the relative abundance (proportions) of each cell type 
# per spot based on the deconvolution results
adata_vis = compute_proportion(adata_vis)

# Perform neighborhood-based clustering on the cell type proportions 
# to identify distinct spatial regions (niches) within the tissue
adata_vis = cluster_spots_by_proportion(adata_vis, n_neighbors=10, resolution=0.3)

# Define the observation column that stores the resulting cluster assignments
CLUSTER_INFO = "proportion_leiden"

# --- Cluster Distribution Summary ---
# Print and display the number of spots assigned to each identified spatial cluster
print("Cluster sizes:")
display(adata_vis.obs[CLUSTER_INFO].value_counts())

## 3) Build observed/expected expression matrices

Upon completing the `ccisignal` module, you typically obtain two primary output files: `sc_ccisignal.h5ad` and `sp_ccisignal.h5ad`. For this tutorial, these two files are provided as precomputed outputs to allow you to proceed directly with the analysis. Using these datasets, we calculate the expected expression values to compare them against the observed expression data.

In [ ]:
# --- Setup and Identifier Simplification ---
cluster_info = CLUSTER_INFO

# Normalize gene names to a standard format (e.g., handling case sensitivity or prefix removal)
simplify_gene_names(adata_ref_sig, SPECIES)
simplify_gene_names(adata_vis, SPECIES)

# --- Reference Signature Extraction (Gene × Cell Type) ---
# Retrieve the list of cell type factors from the reference model
factor_names = list(adata_ref_sig.uns["mod"]["factor_names"])
# Extract the mean expression per cluster; handle different data structures (DataFrame vs. Array)
inf_raw = adata_ref_sig.varm.get("means_per_cluster_mu_fg", None)

if inf_raw is None:
    inf_aver2 = adata_ref_sig.obs[factor_names].copy()
elif isinstance(inf_raw, pd.DataFrame):
    inf_aver2 = inf_raw.copy()
else:
    inf_aver2 = pd.DataFrame(inf_raw, index=adata_ref_sig.var_names, columns=factor_names)

# Clean column names by stripping common prefixes used in modeling outputs
prefix = "means_per_cluster_mu_fg_"
if all(isinstance(c, str) and c.startswith(prefix) for c in inf_aver2.columns):
    inf_aver2.columns = [c[len(prefix):] for c in inf_aver2.columns]

# --- Feature Alignment ---
# Intersect and filter genes present in both the spatial data and the reference signature
genes = np.intersect1d(adata_vis.var_names, inf_aver2.index)
adata_vis = adata_vis[:, genes].copy()
inf_aver2 = inf_aver2.loc[genes]

# Align cell types between spatial deconvolution and reference signature
vis_factors = list(adata_vis.uns["mod"]["factor_names"])
common = [ct for ct in vis_factors if ct in inf_aver2.columns]
if not common:
    raise ValueError(f"No overlapping celltypes. vis={vis_factors}, ref={list(inf_aver2.columns)}")

# Ensure cell abundance estimates (from deconvolution) are accessible in the obs table
if not all(ct in adata_vis.obs.columns for ct in common):
    adata_vis.obs[vis_factors] = pd.DataFrame(
        adata_vis.obsm["q05_cell_abundance_w_sf"], index=adata_vis.obs_names, columns=vis_factors
    )

# --- Calculating Expected Spatial Expression ---
# Use posterior sample means to reconstruct the expected expression based on cell composition
post = adata_vis.uns["mod"]["post_sample_means"]
# dot product: Cell Abundance (Spot x CT) @ Reference Signatures (CT x Gene)
total_df = adata_vis.obs[common] @ inf_aver2[common].T

# Apply technical scaling factors: gene-specific additive effects (s_g) and spot detection efficiency (y_s)
final_df = (total_df.mul(post["m_g"], axis=1) + np.asarray(post["s_g_gene_add"])[0]).mul(post["detection_y_s"], axis=0)

# --- Preparing Dataframes for Comparison ---
meta_data = adata_vis.obs.copy()
# Combine expression values with spot metadata for both Observed and Expected datasets
observed_expression = pd.concat([adata_vis.to_df(), meta_data], axis=1)
expected_expression = pd.concat([final_df, meta_data], axis=1)

# Assign labels and adjust indices to prevent overlap during concatenation
observed_expression["condition"] = "observed"
expected_expression["condition"] = "expected"
observed_expression.index = [f"{i}_obs" for i in observed_expression.index]
expected_expression.index = [f"{i}_exp" for i in expected_expression.index]

# Create a combined AnnData object for statistical testing (Observed vs. Expected)
combined_df = pd.concat([observed_expression, expected_expression])
drop_cols = list(meta_data.columns) + ["condition"]
adata_tests = sc.AnnData(X=combined_df.drop(columns=drop_cols))
adata_tests.obs["condition"] = combined_df["condition"].values
adata_tests.obs[cluster_info] = combined_df[cluster_info].astype("category")

# --- Final Cleanup ---
# Sanitize all column and cell type names for compatibility with downstream analysis tools
observed_expression = clean_column_names(observed_expression)
expected_expression = clean_column_names(expected_expression)
meta_data = clean_column_names(meta_data)
inf_aver2 = clean_column_names(inf_aver2)
cell_types = obtain_clean_celltype_names(adata_vis)

print("OK:", observed_expression.shape, expected_expression.shape, "celltypes used:", len(common))

## 4) Detect spatial region-specific CCI genes (`ccigenes` module)

To identify genes driven by cell-cell interactions (CCI), we employ a robust statistical framework that integrates two distinct approaches: a `permutation test` to evaluate expression differences across spatial regions and a `chi-square test` to assess variance within each region. The results from both tests are combined using the `Cauchy combination test` to determine overall statistical significance. Final CCI gene candidates are then filtered based on adjusted `p-values` and `log-fold change` (LogFC) thresholds.

Note: This analytical pipeline is consistent across various datasets; only the `SPECIES` and `CLUSTER_INFO` parameters require adjustment to match the specific sample metadata.

In [ ]:
# --- Statistical Testing for CCI Gene Identification ---

# Run permutation test across all spatial clusters to assess 
# expression differences between observed and expected data
perm_results = permutation_test_all_clusters(
    adata_tests,
    cluster_info=cluster_info,
    observed_expression=observed_expression,
    expected_expression=expected_expression,
    n_permutations=N_PERMUTATIONS,
    use_zeros=True,
    random_seed=42,
    path=str(OUTPUT_DIR),
)

# Perform Chi-square goodness-of-fit test within each cluster 
# to evaluate the variance between observed and expected distributions
chi_results = chi_square_goodness_of_fit(
    adata_tests,
    cluster_info=cluster_info,
    groupby="condition",
    reference="observed",
    target="expected",
    use_zeros=True,
)

# --- Statistical Integration ---

# Merge permutation and chi-square results for each gene-cluster pair
stats_df = pd.merge(perm_results, chi_results, on=["gene", "cluster"], how="inner")

# Combine p-values from both tests using the Cauchy combination test
stats_df["combined_p_value_adj"] = stats_df.apply(compute_combined_p_value, axis=1)

# Save the comprehensive statistics for all tested genes
stats_df.to_csv(OUTPUT_DIR / "allgenes_results.csv", index=False)

# --- Filtering for Significant CCI Genes ---

# Filter genes based on adjusted p-value, higher observed vs. expected expression, 
# and a minimum log-fold change (LogFC) threshold
gene_filter = (
    (stats_df["combined_p_value_adj"] < 0.01)
    & (stats_df["mean_ref"] > stats_df["mean_tgt"])
    & (stats_df["logfc"] > LOG_FC)
)
final_results = stats_df.loc[gene_filter].copy()

# Save the final list of identified CCI genes (niche-specific genes)
final_results.to_csv(OUTPUT_DIR / "ccigenes_results.csv", index=False)

# --- Results Summary and Display ---

# Display the total count and the top 20 CCI gene candidates
print("Niche gene hits:", final_results.shape[0])
display(final_results[["gene", "cluster", "combined_p_value_adj", "logfc"]].head(20))

# Group by spatial cluster and identify the top 10 most significant genes per region
print("Top 10 niche genes per cluster (by adjusted p):")
top_by_cluster = (
    final_results.sort_values(["cluster", "combined_p_value_adj"]) 
    .groupby("cluster")
    .head(10)
)
display(top_by_cluster[["cluster", "gene", "combined_p_value_adj", "logfc"]])

## 5) Identify interacting cell-type pairs (`ccipairs` module)

Following the identification of CCI genes, we determine the specific interacting cell-type pairs through a `regression-based framework` (a two-step modeling strategy). In this step, we regress the residual expression signals against cell-type interaction terms to quantify pairwise effects. Finally, these significant interaction terms are biologically annotated by mapping them to curated ligand-receptor databases(`Omnipath`, `CellChat`, `CellTalk`), bridging the gap between observed spatial signals and established molecular signaling pathways.

In [ ]:
# --- Identify Interacting Cell-Type Pairs (ccipairs) ---

# Check if any significant CCI (niche) genes were found in the previous step
if final_results.empty:
    print("No niche genes detected; skipping ccipairs.")
else:
    # Normalize cell-type deconvolution values so that the sum per spot equals 1
    norm_deconv = meta_data.loc[:, cell_types].div(meta_data.loc[:, cell_types].sum(axis=1), axis=0)
    norm_deconv[cluster_info] = meta_data[cluster_info]
    
    # Calculate the mean cell-type composition for each spatial cluster (niche)
    cluster_summary = norm_deconv.groupby(cluster_info).mean()
    cluster_summary.loc["mean"] = cluster_summary.mean()

    # Data cleanup: Remove duplicate genes from the reference signature and results
    inf_aver2 = inf_aver2[~inf_aver2.index.duplicated(keep="first")]
    final_results = final_results.drop_duplicates(subset=["gene"])

    # --- Two-Step Regression Modeling ---
    # Regress the residual signal (Observed - Expected) onto cell-type interaction terms.
    # We use Elastic Net regularization (alpha=0.5) to handle multi-collinearity.
    models_per_gene, gene_analysis = regress_residual_on_interaction_with_regularization(
        observed_expression=observed_expression,
        expected_expression=expected_expression,
        celltypes=cell_types,
        cell_sig=inf_aver2,
        niche_gene_results=final_results,
        cluster_summary=cluster_summary,
        cluster_info=cluster_info,
        self_interaction=False, # Exclude self-interactions (e.g., CellTypeA-CellTypeA)
        use_zeros=False,
        alpha=0.5,
    )

    # --- Extract Significant Interactions ---
    # Filter for interaction terms that meet the statistical p-value threshold
    interaction_terms = extract_interaction_terms(
        models_per_gene, gene_analysis, p_value_threshold=PVAL_TERM
    )

    if interaction_terms.empty:
        print("No significant interaction terms.")
    else:
        # For each gene in each cluster, keep the top 5 most significant cell-type pairs
        interaction_terms = (
            interaction_terms.groupby(["cluster", "gene"]) 
            .apply(lambda df: df.sort_values("p_value").head(5))
            .reset_index(drop=True)
        )
        
        # --- Biological Validation ---
        # Map the identified cell-type pairs and genes to curated Ligand-Receptor databases
        interaction_terms = compare_database(interaction_terms, species=SPECIES)
        
        # Save final results containing gene, interacting pairs, and database evidence
        interaction_terms.to_csv(OUTPUT_DIR / "ccipairs_results.csv", index=False)

        # Display the most significant validated interaction results
        print("Top interaction terms (head):")
        display(interaction_terms.head(25))